# Relearning Leakage Analysis

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from matplotlib.colors import Normalize

# ============================================================
# Toggle: "1B" or "7B"
# ============================================================
MODEL_SIZE = "7B"

SAVES_DIR = os.path.join(os.environ['HOME'], 'OLMoBenchOutputs', 'saves')

CONFIG = {
    "1B": os.path.join(SAVES_DIR, 'Train_95frozen_FullSubset'),
    "7B": os.path.join(SAVES_DIR, '7B_Train_95frozen_FullSubset'),
}

OUTPUT_DIR = CONFIG[MODEL_SIZE]

FIELD_FILE_MAP = {
    'Birth City': 'Birth_City',
    'Email Address': 'Email_Address',
    'Phone Number': 'Phone_Number',
    "Driver's License": 'Drivers_License',
}

METHODS = ['AlphaEdit', 'MemFlex', 'SimNPO', 'GradDiff_OracleGrad']
NUM_ATTEMPTS = 200

def load_cached(cache_dir, filename):
    path = os.path.join(cache_dir, filename)
    if not os.path.exists(path):
        print(f'  [MISSING] {filename}')
        return None, None
    with open(path, 'rb') as f:
        cached = pickle.load(f)
    return cached['results'], cached['metrics']

## Paper-ready figures

In [ ]:
from matplotlib import rcParams
from matplotlib.colors import Normalize

rcParams['font.family'] = 'monospace'
rcParams['font.monospace'] = ['Inconsolata', 'Consolas', 'DejaVu Sans Mono']
rcParams['font.style'] = 'normal'
rcParams['font.weight'] = 'bold'

TRUE_BLACK = '#000000'
LW = 1.5
FS = 35

METHOD_COLORS = {
    'AlphaEdit':  '#4E79A7',
    'MemFlex':    '#F28E2B',
    'OracleGrad': '#59A14F',
    'SimNPO':     '#E15759',
}
METHOD_ORDER = ['MemFlex', 'AlphaEdit', 'SimNPO', 'OracleGrad']

BAR_WIDTH = 0.65
HEATMAP_FILL = 0.92
HEATMAP_BORDER = 1.5

# ── Layout: shared constants (inches) ──
PITCH = 3.0       # center-to-center distance for bars AND heatmap columns
FIG_H = 10.0      # identical height for every PDF

# Bar-chart margins (inches)
BAR_ML = 2.0      # y-label + tick labels
BAR_MR = 0.2
BAR_MT = 1.2      # headroom for value labels
BAR_MB = 1.5      # x-tick labels

# Heatmap margins (inches)
HEAT_ML = 3.0     # y-tick labels (method names)
HEAT_MR = 1.3     # colorbar + padding
HEAT_MT = 0.5
HEAT_MB = 1.5     # x-tick labels

# ── Assets output directory ──
ASSETS_RELEARN_DIR = os.path.join((os.getcwd() if os.path.basename(os.getcwd()) == 'notebooks' else os.path.join(os.getcwd(), 'notebooks')), 'assets', 'imgs', 'relearning')
os.makedirs(ASSETS_RELEARN_DIR, exist_ok=True)

for target_field, field_file in FIELD_FILE_MAP.items():
    cache_dir = os.path.join(OUTPUT_DIR, 'cached_notebook_files', field_file)
    if not os.path.exists(cache_dir):
        print(f'[SKIP] {target_field}')
        continue

    mem_metrics = {}
    mem_results = {}
    for method in METHODS:
        results, metrics = load_cached(cache_dir, f'results_relearn_memorized_{method}.pkl')
        if results is not None:
            mem_metrics[method] = metrics
            mem_results[method] = results

    if not mem_metrics:
        print(f'[SKIP] {target_field} — no data')
        continue

    print(f'\n{"="*60}')
    print(f'{MODEL_SIZE} | {target_field}')
    print(f'{"="*60}')

    # ── Prepare bar data ──
    rows = []
    for m in METHODS:
        if m in mem_metrics:
            label = m.replace('GradDiff_OracleGrad', 'OracleGrad')
            rows.append({'Method': label, 'val': mem_metrics[m]['% people with leak']})
    plot_df = pd.DataFrame(rows)
    bar_order = [m for m in METHOD_ORDER if m in plot_df['Method'].values]
    bar_df = plot_df.set_index('Method').loc[bar_order].reset_index()

    n_bars = len(bar_df)
    bar_colors = [METHOD_COLORS[m] for m in bar_df['Method']]
    x = np.arange(n_bars)

    # ── Bar chart ──
    bar_ax_w = PITCH * n_bars
    bar_ax_h = FIG_H - BAR_MT - BAR_MB
    bar_fig_w = BAR_ML + bar_ax_w + BAR_MR

    fig, ax = plt.subplots(figsize=(bar_fig_w, FIG_H))
    ax.set_position([BAR_ML / bar_fig_w, BAR_MB / FIG_H,
                     bar_ax_w / bar_fig_w, bar_ax_h / FIG_H])

    ax.bar(x, bar_df['val'], color=bar_colors, edgecolor=TRUE_BLACK,
           linewidth=LW, width=BAR_WIDTH, zorder=3)
    for i, v in enumerate(bar_df['val']):
        ax.text(i, v + 1.5, f'{v:.0f}%', ha='center', va='bottom',
                fontsize=FS, fontweight='bold', color=TRUE_BLACK)

    ax.set_xlim(-0.5, n_bars - 0.5)
    ax.set_ylim(0, 108)
    ax.set_xticks(x)
    ax.set_xticklabels(bar_df['Method'], fontsize=FS, fontweight='bold')
    ax.set_ylabel(f'Success@{NUM_ATTEMPTS}', fontsize=FS, fontweight='bold', labelpad=12)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter())
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    for spine in ('left', 'bottom'):
        ax.spines[spine].set_linewidth(LW)
        ax.spines[spine].set_color(TRUE_BLACK)
    ax.tick_params(axis='y', labelsize=FS - 4, width=LW, length=8, colors=TRUE_BLACK)
    ax.tick_params(axis='x', width=LW, length=8, colors=TRUE_BLACK, pad=6)
    ax.grid(axis='y', linestyle='--', alpha=0.3, linewidth=LW, zorder=0)
    ax.grid(False, axis='x')

    out = os.path.join(ASSETS_RELEARN_DIR, f'paper_relearn_vulnerability_{MODEL_SIZE}_{field_file}.pdf')
    fig.savefig(out, format='pdf')
    print(f'  Saved {out}')
    plt.show()

    # ── Jaccard heatmap ──
    leaked_people = {}
    for method, results in mem_results.items():
        label = method.replace('GradDiff_OracleGrad', 'OracleGrad')
        leaked_people[label] = set(results[results['leaked']]['person_id'].unique())
    method_names = [m for m in METHOD_ORDER if m in leaked_people]
    n = len(method_names)

    if n < 2:
        print(f'  Only {n} method(s) — skipping overlap')
        continue

    mask_upper = np.tril(np.ones((n, n), dtype=bool), k=-1)
    jaccard_matrix = np.zeros((n, n))
    for i, m1 in enumerate(method_names):
        for j, m2 in enumerate(method_names):
            union = len(leaked_people[m1] | leaked_people[m2])
            jaccard_matrix[i, j] = (len(leaked_people[m1] & leaked_people[m2]) / union
                                    if union > 0 else 0)

    heat_ax_w = PITCH * n
    heat_ax_h = FIG_H - HEAT_MT - HEAT_MB
    heat_fig_w = HEAT_ML + heat_ax_w + HEAT_MR

    fig, ax = plt.subplots(figsize=(heat_fig_w, FIG_H))
    ax.set_position([HEAT_ML / heat_fig_w, HEAT_MB / FIG_H,
                     heat_ax_w / heat_fig_w, heat_ax_h / FIG_H])

    cmap_obj = plt.get_cmap('BuPu')
    norm = Normalize(vmin=0, vmax=1)

    ax.set_facecolor('white')
    ax.set_xlim(0, n)
    ax.set_ylim(n, 0)

    gap_frac = 1 - HEATMAP_FILL
    pad = gap_frac / 2

    for i in range(n):
        for j in range(n):
            if not mask_upper[i, j]:
                val = jaccard_matrix[i, j]
                color = cmap_obj(norm(val))
                ax.add_patch(plt.Rectangle(
                    (j, i), 1, 1,
                    fill=True, facecolor='white', edgecolor='none', zorder=2))
                ax.add_patch(plt.Rectangle(
                    (j + pad, i + pad), 1 - 2 * pad, 1 - 2 * pad,
                    fill=True, facecolor=color, edgecolor=TRUE_BLACK,
                    linewidth=HEATMAP_BORDER, zorder=3))
                lum = 0.299 * color[0] + 0.587 * color[1] + 0.114 * color[2]
                txt_color = 'white' if lum < 0.5 else '#333333'
                ax.text(j + 0.5, i + 0.5, f'{val:.2f}',
                        ha='center', va='center', fontsize=FS,
                        fontweight='bold', color=txt_color, zorder=4)

    ax.set_xticks([k + 0.5 for k in range(n)])
    ax.set_xticklabels(method_names, fontsize=FS, fontweight='bold')
    ax.set_yticks([k + 0.5 for k in range(n)])
    ax.set_yticklabels(method_names, fontsize=FS, fontweight='bold')
    ax.tick_params(axis='both', which='both', length=0, pad=6)
    for spine in ax.spines.values():
        spine.set_visible(False)

    # Colorbar
    sm = plt.cm.ScalarMappable(cmap=cmap_obj, norm=norm)
    sm.set_array([])
    cbar_x0 = (HEAT_ML + heat_ax_w + 0.15) / heat_fig_w
    cbar_w = 0.2 / heat_fig_w
    cbar_y0 = HEAT_MB / FIG_H + 0.1 * (heat_ax_h / FIG_H)
    cbar_h = 0.8 * (heat_ax_h / FIG_H)
    cbar_ax = fig.add_axes([cbar_x0, cbar_y0, cbar_w, cbar_h])
    cbar = fig.colorbar(sm, cax=cbar_ax)
    cbar.ax.tick_params(labelsize=FS - 6)

    out = os.path.join(ASSETS_RELEARN_DIR, f'paper_relearn_overlap_{MODEL_SIZE}_{field_file}.pdf')
    fig.savefig(out, format='pdf')
    print(f'  Saved {out}')
    plt.show()

    # Verify pitch match
    for a, label in [(ax, 'Heatmap')]:
        fig.canvas.draw()
        p0 = a.transData.transform((0, 0))
        p1 = a.transData.transform((1, 0))
        actual_pitch = (p1[0] - p0[0]) / fig.dpi
        print(f'  {label} pitch: {actual_pitch:.3f} in/unit (target {PITCH:.3f})')

    for m, s in leaked_people.items():
        print(f'  {m}: {len(s)} people leaked')

## Combined 1B + 7B "thirds" figures
Each field produces **3 PDFs** (bar chart + Jaccard×2), all the same figure size so they tile at `width=0.33\textwidth` in LaTeX.

In [ ]:
# ── Combined 1B + 7B "thirds" layout ──
# Output per field:
#   paper_relearn_vulnerability_thirds_{field}.pdf   (grouped bar chart)
#   paper_relearn_overlap_thirds_1B_{field}.pdf      (Jaccard 1B)
#   paper_relearn_overlap_thirds_7B_{field}.pdf      (Jaccard 7B)

ABBREV_MAP = {
    'AlphaEdit': 'Alpha\nEdit',
    'MemFlex':   'Mem\nFlex',
    'SimNPO':    'Sim\nNPO',
    'OracleGrad':'Oracle\nGrad',
}
# Unified order for BOTH bar chart and Jaccard matrices: AE, MF, SN, OG
THIRDS_METHOD_ORDER = ['AlphaEdit', 'MemFlex', 'SimNPO', 'OracleGrad']

# ── Shared dimensions (inches) ── all three PDFs are the same size
THIRDS_W  = 10.0
THIRDS_H  = FIG_H          # reuse 10.0

# ── Shared top/bottom margins so plots align vertically in LaTeX ──
T_MT = 0.8        # top  margin (headroom for bar labels)
T_MB = 1.3        # bottom margin (x-tick labels)

# Bar-chart left/right margins
TB_ML, TB_MR = 2.0, 0.15

# Heatmap left/right margins (abbreviated labels left, colorbar right)
TH_ML, TH_MR = 1.9, 1.2

# Sub-bar geometry — wider bars + bigger gap to prevent label overlap
SUB_W   = 0.30                    # width of each sub-bar
SUB_GAP = 0.08                    # gap between 1B and 7B bars
ALPHA_1B = 1.0
ALPHA_7B = 0.45

def center_multiline_labels(labels):
    """Set multialignment='center' on tick labels so \\n-split lines are centered."""
    for lbl in labels:
        lbl.set_multialignment('center')

# ── Assets output directory ──
ASSETS_RELEARN_DIR = os.path.join((os.getcwd() if os.path.basename(os.getcwd()) == 'notebooks' else os.path.join(os.getcwd(), 'notebooks')), 'assets', 'imgs', 'relearning')
os.makedirs(ASSETS_RELEARN_DIR, exist_ok=True)
# "thirds" PDFs go in a dedicated subdirectory to match the paper's \includegraphics paths
THIRDS_DIR = os.path.join(ASSETS_RELEARN_DIR, 'thirds')
os.makedirs(THIRDS_DIR, exist_ok=True)

for target_field, field_file in FIELD_FILE_MAP.items():

    # ── Load both model sizes ──
    all_data = {}
    for size in ['1B', '7B']:
        cache_dir = os.path.join(CONFIG[size], 'cached_notebook_files', field_file)
        if not os.path.exists(cache_dir):
            continue
        mm, mr = {}, {}
        for method in METHODS:
            results, metrics = load_cached(cache_dir,
                                           f'results_relearn_memorized_{method}.pkl')
            if results is not None:
                label = method.replace('GradDiff_OracleGrad', 'OracleGrad')
                mm[label] = metrics
                mr[label] = results
        if mm:
            all_data[size] = {'metrics': mm, 'results': mr}

    if not all_data:
        print(f'[SKIP] {target_field} — no data')
        continue

    print(f'\n{"="*60}')
    print(f'THIRDS | {target_field}')
    print(f'{"="*60}')

    # ── Methods present in at least one size, in AE/MF/SN/OG order ──
    bar_order = [m for m in THIRDS_METHOD_ORDER
                 if any(m in all_data.get(s, {}).get('metrics', {})
                        for s in ['1B', '7B'])]
    bar_labels = [ABBREV_MAP[m] for m in bar_order]
    n_bars = len(bar_order)
    x = np.arange(n_bars)

    # ====================================================================
    # 1) GROUPED BAR CHART
    # ====================================================================
    ax_w = THIRDS_W - TB_ML - TB_MR
    ax_h = THIRDS_H - T_MT - T_MB

    fig, ax = plt.subplots(figsize=(THIRDS_W, THIRDS_H))
    ax.set_position([TB_ML / THIRDS_W, T_MB / THIRDS_H,
                     ax_w  / THIRDS_W, ax_h / THIRDS_H])

    for size, alpha, sign in [('1B', ALPHA_1B, -1), ('7B', ALPHA_7B, +1)]:
        if size not in all_data:
            continue
        offset = sign * (SUB_W / 2 + SUB_GAP / 2)
        vals = [all_data[size]['metrics'].get(m, {}).get('% people with leak', 0)
                for m in bar_order]
        colors = [METHOD_COLORS[m] for m in bar_order]
        ax.bar(x + offset, vals, SUB_W,
               color=colors, edgecolor=TRUE_BLACK, linewidth=LW,
               alpha=alpha, zorder=3)
        for i, v in enumerate(vals):
            ax.text(x[i] + offset, v + 1.5, f'{v:.0f}%',
                    ha='center', va='bottom',
                    fontsize=FS - 6, fontweight='bold', color=TRUE_BLACK)

    ax.set_xlim(-0.5, n_bars - 0.5)
    ax.set_ylim(0, 108)
    ax.set_xticks(x)
    ax.set_xticklabels(bar_labels, fontsize=FS, fontweight='bold')
    center_multiline_labels(ax.get_xticklabels())
    ax.set_ylabel(f'Success@{NUM_ATTEMPTS}', fontsize=FS, fontweight='bold', labelpad=12)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter())
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    for spine in ('left', 'bottom'):
        ax.spines[spine].set_linewidth(LW)
        ax.spines[spine].set_color(TRUE_BLACK)
    ax.tick_params(axis='y', labelsize=FS - 4, width=LW, length=8, colors=TRUE_BLACK)
    ax.tick_params(axis='x', width=LW, length=8, colors=TRUE_BLACK, pad=6)
    ax.grid(axis='y', linestyle='--', alpha=0.3, linewidth=LW, zorder=0)
    ax.grid(False, axis='x')

    # Legend (dark = 1B, light = 7B)
    h1 = plt.Rectangle((0, 0), 1, 1, fc='grey', ec=TRUE_BLACK, lw=LW, alpha=ALPHA_1B)
    h7 = plt.Rectangle((0, 0), 1, 1, fc='grey', ec=TRUE_BLACK, lw=LW, alpha=ALPHA_7B)
    ax.legend([h1, h7], ['1B', '7B'], fontsize=FS - 8, frameon=False, loc='upper right')

    out = os.path.join(THIRDS_DIR,
                       f'paper_relearn_vulnerability_thirds_{field_file}.pdf')
    fig.savefig(out, format='pdf')
    print(f'  Saved {out}')
    plt.show()

    # ====================================================================
    # 2) JACCARD HEATMAPS (one per model size)
    # ====================================================================
    for size in ['1B', '7B']:
        if size not in all_data:
            continue

        leaked_people = {}
        for method, results in all_data[size]['results'].items():
            leaked_people[method] = set(
                results[results['leaked']]['person_id'].unique())

        method_names = [m for m in THIRDS_METHOD_ORDER if m in leaked_people]
        n = len(method_names)
        if n < 2:
            print(f'  {size}: only {n} method(s) — skipping overlap')
            continue

        # Jaccard matrix (full, upper-triangle shown)
        mask_upper = np.tril(np.ones((n, n), dtype=bool), k=-1)
        jac = np.zeros((n, n))
        for i, m1 in enumerate(method_names):
            for j, m2 in enumerate(method_names):
                u = len(leaked_people[m1] | leaked_people[m2])
                jac[i, j] = (len(leaked_people[m1] & leaked_people[m2]) / u
                             if u > 0 else 0)

        abbrev = [ABBREV_MAP[m] for m in method_names]

        ax_w = THIRDS_W - TH_ML - TH_MR
        ax_h = THIRDS_H - T_MT - T_MB

        fig, ax = plt.subplots(figsize=(THIRDS_W, THIRDS_H))
        ax.set_position([TH_ML / THIRDS_W, T_MB / THIRDS_H,
                         ax_w  / THIRDS_W, ax_h / THIRDS_H])

        cmap_obj = plt.get_cmap('BuPu')
        norm = Normalize(vmin=0, vmax=1)

        ax.set_facecolor('white')
        ax.set_xlim(0, n)
        ax.set_ylim(n, 0)

        gap_frac = 1 - HEATMAP_FILL
        pad = gap_frac / 2

        for i in range(n):
            for j in range(n):
                if not mask_upper[i, j]:
                    val = jac[i, j]
                    color = cmap_obj(norm(val))
                    ax.add_patch(plt.Rectangle(
                        (j, i), 1, 1,
                        fill=True, facecolor='white', edgecolor='none', zorder=2))
                    ax.add_patch(plt.Rectangle(
                        (j + pad, i + pad),
                        1 - 2 * pad, 1 - 2 * pad,
                        fill=True, facecolor=color, edgecolor=TRUE_BLACK,
                        linewidth=HEATMAP_BORDER, zorder=3))
                    lum = 0.299*color[0] + 0.587*color[1] + 0.114*color[2]
                    txt_c = 'white' if lum < 0.5 else '#333333'
                    ax.text(j + 0.5, i + 0.5, f'{val:.2f}',
                            ha='center', va='center', fontsize=FS,
                            fontweight='bold', color=txt_c, zorder=4)

        ax.set_xticks([k + 0.5 for k in range(n)])
        ax.set_xticklabels(abbrev, fontsize=FS, fontweight='bold')
        center_multiline_labels(ax.get_xticklabels())
        ax.set_yticks([k + 0.5 for k in range(n)])
        ax.set_yticklabels(abbrev, fontsize=FS, fontweight='bold')
        center_multiline_labels(ax.get_yticklabels())
        ax.tick_params(axis='both', which='both', length=0, pad=6)
        for spine in ax.spines.values():
            spine.set_visible(False)

        # Colorbar
        sm = plt.cm.ScalarMappable(cmap=cmap_obj, norm=norm)
        sm.set_array([])
        cbar_x0 = (TH_ML + ax_w + 0.15) / THIRDS_W
        cbar_w  = 0.2  / THIRDS_W
        cbar_y0 = T_MB / THIRDS_H + 0.1 * (ax_h / THIRDS_H)
        cbar_h  = 0.8 * (ax_h / THIRDS_H)
        cbar_ax = fig.add_axes([cbar_x0, cbar_y0, cbar_w, cbar_h])
        cbar = fig.colorbar(sm, cax=cbar_ax)
        cbar.ax.tick_params(labelsize=FS - 6)

        out = os.path.join(THIRDS_DIR,
                           f'paper_relearn_overlap_thirds_{size}_{field_file}.pdf')
        fig.savefig(out, format='pdf')
        print(f'  Saved {out}')
        plt.show()

        for m, s in leaked_people.items():
            print(f'  {size} {m}: {len(s)} people leaked')

In [ ]:
# ── Leaked people count: grouped bar chart (methods × fields) ──

# Collect counts
count_rows = []
for target_field, field_file in FIELD_FILE_MAP.items():
    cache_dir = os.path.join(OUTPUT_DIR, 'cached_notebook_files', field_file)
    if not os.path.exists(cache_dir):
        continue
    for method in METHODS:
        results, metrics = load_cached(cache_dir, f'results_relearn_memorized_{method}.pkl')
        if results is None:
            continue
        label = method.replace('GradDiff_OracleGrad', 'OracleGrad')
        n_leaked = results[results['leaked']]['person_id'].nunique()
        total = results['person_id'].nunique()
        count_rows.append({
            'Field': target_field, 'Method': label,
            'Leaked': n_leaked, 'Total': total,
        })

count_df = pd.DataFrame(count_rows)

if not count_df.empty:
    fields_present = list(FIELD_FILE_MAP.keys())
    methods_present = [m for m in METHOD_ORDER if m in count_df['Method'].values]
    n_fields = len(fields_present)
    n_methods = len(methods_present)

    bar_w = 0.18
    x = np.arange(n_fields)

    fig, ax = plt.subplots(figsize=(max(14, 3.5 * n_fields), 8))

    for j, method in enumerate(methods_present):
        offset = (j - (n_methods - 1) / 2) * bar_w
        vals = []
        for field in fields_present:
            row = count_df[(count_df['Field'] == field) & (count_df['Method'] == method)]
            vals.append(row['Leaked'].values[0] if len(row) > 0 else 0)
        bars = ax.bar(x + offset, vals, bar_w,
                      color=METHOD_COLORS[method], edgecolor=TRUE_BLACK,
                      linewidth=LW, label=method, zorder=3)
        for i, v in enumerate(vals):
            if v > 0:
                ax.text(x[i] + offset, v + 0.5, str(v), ha='center', va='bottom',
                        fontsize=14, fontweight='bold', color=TRUE_BLACK)

    ax.set_xticks(x)
    ax.set_xticklabels(fields_present, fontsize=18, fontweight='bold')
    ax.set_ylabel('# Leaked People', fontsize=20, fontweight='bold', labelpad=12)
    ax.set_title(f'{MODEL_SIZE} — Leaked People per Method & Field', fontsize=22, fontweight='bold', pad=15)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    for spine in ('left', 'bottom'):
        ax.spines[spine].set_linewidth(LW)
        ax.spines[spine].set_color(TRUE_BLACK)
    ax.tick_params(axis='y', labelsize=16, width=LW, length=8, colors=TRUE_BLACK)
    ax.tick_params(axis='x', width=LW, length=8, colors=TRUE_BLACK, pad=6)
    ax.grid(axis='y', linestyle='--', alpha=0.3, linewidth=LW, zorder=0)
    ax.grid(False, axis='x')
    ax.legend(fontsize=16, frameon=False, loc='upper right')

    fig.tight_layout()
    plt.show()

    # Also print the summary table
    pivot = count_df.pivot(index='Method', columns='Field', values='Leaked')
    pivot = pivot.reindex(index=methods_present, columns=fields_present)
    display(pivot)